# 13 · Tuned Distribution-Balanced Loss screen

Notebook 12 kết luận: chỉ `WASL α=.50` qua được cổng "promising", nhưng **overfit rất nhanh**
(tail mAP 0.1230 @ep2 → 0.1098 @ep8, sụt 0.0132). Trước khi replicate 3 seed, chạy một screen
DB-Loss có tuning đàng hoàng.

Kết quả nb12 (val, seed 42) làm mốc:

| Loss | ep | Macro mAP | Tail mAP | Worst-3 fixed | Tail Recall@K |
|---|---:|---:|---:|---:|---:|
| ASL | 3 | 0.4406 | 0.1210 | 0.1370 | 0.1963 |
| **WASL .50** | **2** | 0.4405 | **0.1230** | **0.1388** | 0.1963 |

## Nguyên tắc (§2)

Kết quả cũ "ASL > DB Loss" chỉ có nghĩa **dưới cấu hình DB đã dùng lúc đó**, không phải
"rebalancing phân phối vô dụng cho cohort này". Screen này tune **chính cơ chế re-balancing
của DBL**, cụ thể là vị trí `mu` trong

$$\hat r = \alpha_{map} + \sigma[\beta_{map}(r-\mu)]$$

**KHÔNG** nhân DBL với weight nghịch-tần ngoài, **KHÔNG** tune `loss_weight` scalar,
**KHÔNG** đổi sampler.

## 5 arm (seed 42, 8 epoch cố định, không early stop, không fork)

| arm | loss | map α | map β | mu | NTR |
|---|---|---:|---:|---:|---|
| `A0` | ASL anchor | — | — | — | — |
| `D0` | DB-Focal **legacy** | 0.1 | 10 | **0.90** | neg_scale 2, init_bias .05 |
| `D1` | DB-Focal | 0.1 | 10 | 0.10 | neg_scale 2, init_bias .05 |
| `D2` | DB-Focal | 0.1 | 10 | 0.20 | neg_scale 2, init_bias .05 |
| `D3` | DB-Focal | 0.1 | 10 | 0.30 | neg_scale 2, init_bias .05 |

`A0` vừa là mốc paired seed-42, vừa là **regression test**: nó phải tái hiện quỹ đạo `L0` của
nb12 (Macro ~0.4406, Tail ~0.1210 @ep3). Lệch nhiều ⇒ có regression ở code-path/data-order.

Không cần fork vì cả 5 arm khác nhau ngay từ epoch 1.

## Đóng băng (§4)

Mọi thứ ngoài loss lấy **verbatim** từ nb12: architecture, H5 warm-start, adapter block 10–11,
Full-EHR encoder, split, natural sampler, augmentation, batch/EFF 256, AdamW 1e-4/1e-5, wd,
scheduler, decoder freeze ep1 → unfreeze, L2-SP, grad clip, pool 7:3, 8 epoch, no early stop.
Test không đụng tới.

`Worst-3` là **tập nhãn khoá** `Fracture, Pleural_Other, Enlarged_Cardiomediastinum` (§10) —
không định nghĩa lại từ arm DBL.

## C0 · Config + preregistration (DBL screen)

In [ ]:
import os, json, math, time, copy, pickle, random, shutil, tarfile, contextlib
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
LOCAL_IMG = os.environ.get('LOCAL_IMG', '/content/cxr_local')
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
BB_CKPT   = os.path.join(DATA_DIR, 'lpft_v2', 'raddino_mld_db_ft_best.pt')            # 04b backbone
H5_CKPT   = os.path.join(DATA_DIR, 'token_fusion', 'checkpoints', 'H5_s42_best.pt')   # H5 head (06b)
NB12_DIR  = os.path.join(DATA_DIR, 'loss_screen')                                     # de doi chieu A0
SC_DIR    = os.path.join(DATA_DIR, 'dbl_screen')
SNAP_DIR  = f'{SC_DIR}/snapshots'
for _d in (SC_DIR, f'{SC_DIR}/checkpoints', f'{SC_DIR}/metrics', SNAP_DIR): os.makedirs(_d, exist_ok=True)

W_F, W_L = 0.7, 0.3
SEED     = 42

# --- FREEZE: y het nb12 / nb11 (late Full-EHR L_E) ---
LATE_1IDX = (10, 11)
LATE      = tuple(b-1 for b in LATE_1IDX)
FREEZE_EHR_ENCODER    = False
DECODER_WARMUP_EPOCHS = 1
LR_ADAPTER            = 1e-4
LR_DECODER            = 1e-5
L2SP_DECODER          = 1e-5
CLIP_DECODER, CLIP_OTHER = 1.0, 5.0
EFF, EPOCHS = 256, 8
MACRO_TOL   = 0.0015                      # §11, y het nb12

# --- BIEN THI NGHIEM DUY NHAT: co che re-balancing cua DBL (mu) ---
ARMS      = ['A0', 'D0', 'D1', 'D2', 'D3']
DB_MU     = {'D0': 0.90, 'D1': 0.10, 'D2': 0.20, 'D3': 0.30}   # D0 = legacy, xac nhan o C8
ARM_LABEL = {'A0':'ASL', 'D0':'DBL mu=.90 (legacy)', 'D1':'DBL mu=.10',
             'D2':'DBL mu=.20', 'D3':'DBL mu=.30'}
WORST3_FIXED = ['Fracture', 'Pleural_Other', 'Enlarged_Cardiomediastinum']            # §10 KHOA

# --- moc nb12 de so sanh (§14); doc tu file neu co, khong thi dung so trong plan ---
NB12_REF = {'ASL':      dict(epoch=3, macro_map=0.4406, tail_map=0.1210, worst3_fixed=0.1370, tail_recall_k=0.1963),
            'WASL .50': dict(epoch=2, macro_map=0.4405, tail_map=0.1230, worst3_fixed=0.1388, tail_recall_k=0.1963)}
_p = f'{NB12_DIR}/metrics/stage1_result.json'
if os.path.exists(_p):
    _r = json.load(open(_p))['selected']
    NB12_REF = {'ASL': _r['L0'], 'WASL .50': _r['L2']}
    print('moc nb12 <- stage1_result.json (so that tu run)')
else:
    print('moc nb12 <- hang so trong plan (khong thay stage1_result.json)')

PREREG = dict(
    experiment='Stage-1A tuned DB-Loss screen: tune re-balancing mapping mu only',
    architecture='FROZEN = nb12 / nb11 L_E (late Full-EHR, adapters block 10,11 1-idx)',
    only_variable="DBL map_param['gamma'] (= paper mu). map alpha/beta, NTR, focal deu CO DINH",
    arms=ARMS, db_mu=DB_MU, anchor='A0 = ASL, paired seed-42 reference + regression test',
    fork='none (moi arm khac nhau tu epoch 1)',
    map_alpha=0.1, map_beta=10.0, neg_scale=2.0, init_bias=0.05,
    focal='giu nguyen ho DB-Focal cua legacy (xem C8 audit)',
    freq_source='study-level TRAIN positives, recompute + cross-check class_freq.pkl',
    sampler='natural (KHONG dung class-aware sampler)',
    seed=SEED, epochs=EPOCHS, early_stopping=False,
    late_blocks_1idx=list(LATE_1IDX), d_e=128, bottleneck=64,
    lr_adapter=LR_ADAPTER, lr_decoder=LR_DECODER, l2sp=L2SP_DECODER,
    decoder_warmup_epochs=DECODER_WARMUP_EPOCHS, wd=1e-2, warmup_frac=0.05,
    effective_batch=EFF, features='full-EHR (labs+micro+vitals+como)', decoder='delayed_unfreeze',
    eval='VAL only, study-level pool 7:3, NO TEST',
    worst3_fixed=WORST3_FIXED,
    primary=['tail_map', 'worst3_fixed', 'tail_recall_k'], safeguard='macro_map',
    ckpt_rule=f'macro >= best-{MACRO_TOL} -> max tail_map (tie: worst3_fixed, recall@K, macro)',
    go_no_go='d_tail_map >= +0.002 AND d_macro_map >= -0.002 AND d_worst3_fixed > -0.002',
    stop='sau Stage-1A DUNG; Stage-1B (neg_scale) chi chay neu co arm qua cong')
json.dump(PREREG, open(f'{SC_DIR}/preregistration.json','w'), indent=1)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '|', torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU')
if dev=='cuda': print(f'  VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
print(f'arms {ARMS} | mu {DB_MU} | seed {SEED} | {EPOCHS} epoch, no early stop, no fork')
print('PREREG khoa ->', f'{SC_DIR}/preregistration.json')

## C0-bis · Stage ảnh (Colab). No-op nếu ảnh đã ở `LOCAL_IMG`.

In [ ]:
os.makedirs(LOCAL_IMG, exist_ok=True)
_have = len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')])
if _have > 1000 and not os.path.isdir(SHARD_DIR):
    print(f'{_have:,} anh da co san o {LOCAL_IMG}; khong co SHARD_DIR -> bo qua staging')
else:
    tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar')) if os.path.isdir(SHARD_DIR) else []
    assert tars or _have > 1000, f'khong co shard o {SHARD_DIR} va {LOCAL_IMG} rong -> can data'
    todo = [f for f in tars if not os.path.exists(os.path.join(LOCAL_IMG, f'.done_{f}'))]
    print(f'shards: {len(tars)} | to stage: {len(todo)}')
    for f in tqdm(todo, desc='staging shards', unit='shard'):
        tmp = f'/content/{f}'
        shutil.copyfile(os.path.join(SHARD_DIR, f), tmp)
        with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)
        os.remove(tmp); open(os.path.join(LOCAL_IMG, f'.done_{f}'), 'w').close()
    print('local images:', len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')]))

## C1 · Cohort + **Full EHR** data-prep (verbatim 11·C1) + tier khóa

Tier lấy y nguyên định nghĩa cũ (`class_freq.pkl`, prevalence train < 0.02 = tail,
< 0.10 = medium). **Không** định nghĩa lại sau khi thấy kết quả.

In [ ]:
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; assert len(LABELS) == 13
ehr = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet')).sort_values('study_id').reset_index(drop=True)
N = len(ehr)
Y_STUDY  = ehr[LABELS].to_numpy('float32')
EHR_ROW  = {int(s): i for i, s in enumerate(ehr.study_id.to_numpy())}

# ---- vocab + full-EHR arrays (verbatim 06b/11) ----
vocab = pickle.load(open(os.path.join(OUT_DIR, 'ehr_vocab.pkl'), 'rb'))
ORG_V, SPEC_V = vocab['org_vocab_size'], vocab['spec_vocab_size']
MISSING = vocab['reserved']['MISSING']

def as_id_lists(col):
    out = []
    for v in ehr[col].to_numpy():
        a = np.asarray(v, np.int64).ravel() if v is not None and np.ndim(v) else np.array([], np.int64)
        out.append(a if a.size else np.array([MISSING], np.int64))
    return out
RAGGED = {'t1_org':('t1_org_ids',ORG_V), 't1_spec':('t1_spec_ids',SPEC_V),
          't2_spec':('t2_spec_ids',SPEC_V), 't3_org':('t3_org_ids',ORG_V)}
bags = {k: as_id_lists(c) for k,(c,_) in RAGGED.items()}

def codes(series, cats):
    m = {c.lower(): i for i, c in enumerate(cats)}
    return series.astype(str).str.lower().map(lambda x: m.get(x, 0)).to_numpy('int64'), len(cats)
set_id,  _ = codes(ehr['setting'],        ['inpatient','ED','outpatient'])
rec_id,  _ = codes(ehr['t3_recency_bin'], ['none','7-30d','30-90d'])
stat_id    = ehr['t1_status'].to_numpy('int64')

BIN = ['t2_pending','t2_time_imprecise','t1_same_encounter']
binv = {b: ehr[b].to_numpy('float32') for b in BIN}
tr_mask = (ehr.split == 'train').to_numpy()
def zscore_train(x, use_log=True):
    z = np.log1p(np.clip(x.astype('float32'), 0, None)) if use_log else x.astype('float32')
    mu, sd = z[tr_mask].mean(), z[tr_mask].std() or 1.0
    return ((z - mu)/sd).astype('float32')
COUNTS = ['t2_n_pending','t3_n_specimens','t4_n_prior_positive_ever',
          't4_years_since_first_culture','t4_n_prior_encounters']
cnts = {c: zscore_train(ehr[c].to_numpy()) for c in COUNTS}

_z = lambda k: np.zeros((N, k), 'float32')
LABS   = vocab['labs']
lab_val  = np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in LABS], 1)
lab_mask = np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in LABS], 1)
VITALS = [c for c in vocab.get('vitals', []) if f'{c}_value' in ehr.columns]
vit_val  = np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in VITALS], 1) if VITALS else _z(0)
vit_mask = np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in VITALS], 1) if VITALS else _z(0)
COMO_COLS = [c for c in (vocab.get('comorbidity', []) + ['como_mask']) if c in ehr.columns]
como_arr  = np.stack([ehr[c].to_numpy('float32') for c in COMO_COLS], 1) if COMO_COLS else _z(0)
REC1 = ['t1_days_since_last_specimen','t1_days_since_last_positive']; REC3 = ['t3_days_since_last_positive']
rec_block = lambda ns: (np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in ns], 1),
                        np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in ns], 1))
rec1_v, rec1_m = rec_block(REC1); rec3_v, rec3_m = rec_block(REC3)
MICRO_TIERS = (1, 2, 3)

cf = pickle.load(open(os.path.join(OUT_DIR, 'class_freq.pkl'), 'rb'))
N_POS_TRAIN = np.asarray(cf['class_freq'], dtype='float64')     # study-level TRAIN positives
N_TRAIN     = int(cf['num_train'])
TRAIN_PREV  = N_POS_TRAIN / N_TRAIN
TIER = np.where(TRAIN_PREV<0.02,'tail', np.where(TRAIN_PREV<0.10,'medium','head'))
TAIL_IDX = np.where(TIER=='tail')[0]
print(f'cohort {N:,} | labs {len(LABS)} | vitals {len(VITALS)} | como {COMO_COLS} | '
      f'micro org/spec {ORG_V}/{SPEC_V} | tiers {MICRO_TIERS}')
print(f'train studies {N_TRAIN:,} | tiers nhan: {dict(zip(*np.unique(TIER,return_counts=True)))}')
print('tail labels:', [LABELS[j] for j in TAIL_IDX])

vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal','lateral'])]
_dup = [c for c in ('subject_id','split','setting') if c in vm.columns]
man = vm.drop(columns=_dup).merge(ehr[['study_id','subject_id','split']], on='study_id', how='inner')
before = len(man)
man = man[man.dicom_id.map(lambda d: os.path.exists(f'{LOCAL_IMG}/{d}.jpg'))].sort_values('dicom_id').reset_index(drop=True)
assert len(man), f'ALL {before:,} images missing -> chay C0-bis'
assert (before-len(man))/before < 0.01, f'{before-len(man)} images missing -> chay C0-bis'
man['ehr_row'] = man.study_id.map(EHR_ROW).astype('int64')
IDX = {s: np.where(man.split.to_numpy()==s)[0] for s in ('train','val','test')}
print('images by split:', {s: int(len(IDX[s])) for s in IDX})
print('NOTE: test index ton tai nhung KHONG BAO GIO dung trong screen nay.')

# ---- §6.1 nguon tan suat cho DBL: recompute study-level TRAIN + doi chieu class_freq.pkl ----
_tr = (ehr.split == 'train').to_numpy()
N_POS_RECOMP = ehr.loc[_tr, LABELS].to_numpy('float64').sum(0)
assert int(_tr.sum()) == N_TRAIN, f'train studies {int(_tr.sum()):,} != class_freq num_train {N_TRAIN:,}'
assert np.array_equal(N_POS_RECOMP, N_POS_TRAIN), (
    'class_freq.pkl KHAC study-level train counts vua tinh lai:\n'
    f'  pkl      {N_POS_TRAIN.astype(int)}\n  recompute {N_POS_RECOMP.astype(int)}')
print(f'§6.1 freq source OK | {int(_tr.sum()):,} train studies (study-level, KHONG phai image-level) '
      f'| imbalance {N_POS_TRAIN.max()/N_POS_TRAIN.min():.1f}x')

## C2 · Dataset (verbatim 11·C2) — ragged bag `(cat,offsets)`

In [ ]:
from transformers import AutoImageProcessor
from torchvision import transforms as T
from PIL import Image
proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MODEL_SIZE = proc.crop_size['height']; MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(MODEL_SIZE, scale=(0.85,1.0), ratio=(0.95,1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])
eval_tf  = T.Compose([T.CenterCrop(MODEL_SIZE), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])

class ImgDS(Dataset):
    def __init__(self, idx, tf, perm=None): self.idx, self.tf, self.perm = np.asarray(idx), tf, perm
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        r = man.iloc[self.idx[k]]
        im = Image.open(f'{LOCAL_IMG}/{r.dicom_id}.jpg').convert('L')
        im = Image.merge('RGB', (im, im, im))
        tr = int(r.ehr_row)
        e  = tr if self.perm is None else int(self.perm[tr])
        d = dict(image=self.tf(im), y=torch.from_numpy(Y_STUDY[tr]),
                 set_id=set_id[e], rec_id=rec_id[e], stat_id=stat_id[e],
                 lab_val=torch.from_numpy(lab_val[e]), lab_mask=torch.from_numpy(lab_mask[e]),
                 vit_val=torch.from_numpy(vit_val[e]), vit_mask=torch.from_numpy(vit_mask[e]),
                 como=torch.from_numpy(como_arr[e]),
                 rec1_v=torch.from_numpy(rec1_v[e]), rec1_m=torch.from_numpy(rec1_m[e]),
                 rec3_v=torch.from_numpy(rec3_v[e]), rec3_m=torch.from_numpy(rec3_m[e]))
        for _bn in BIN:    d[_bn] = binv[_bn][e]
        for _c in COUNTS:  d[_c]  = cnts[_c][e]
        for nm in RAGGED:  d[nm]  = torch.from_numpy(bags[nm][e])
        return d

def collate(bb):
    o = dict(image=torch.stack([x['image'] for x in bb]), y=torch.stack([x['y'] for x in bb]),
             set_id=torch.tensor([x['set_id'] for x in bb]),
             rec_id=torch.tensor([x['rec_id'] for x in bb]),
             stat_id=torch.tensor([x['stat_id'] for x in bb]))
    for kk in ('lab_val','lab_mask','vit_val','vit_mask','como','rec1_v','rec1_m','rec3_v','rec3_m'):
        o[kk] = torch.stack([x[kk] for x in bb])
    for kk in BIN + COUNTS: o[kk] = torch.tensor([x[kk] for x in bb], dtype=torch.float32)
    for nm in RAGGED:
        lens = [len(x[nm]) for x in bb]
        o[nm] = (torch.cat([x[nm] for x in bb]) if sum(lens) else torch.empty(0, dtype=torch.int64),
                 torch.tensor([0]+lens[:-1]).cumsum(0))
    return o

def loader(split, tf, shuffle, bs, perm=None, workers=8):
    return DataLoader(ImgDS(IDX[split], tf, perm), batch_size=bs, shuffle=shuffle, collate_fn=collate,
                      num_workers=workers, pin_memory=(dev=='cuda'), persistent_workers=(workers>0))
to_dev = lambda b: {k:(tuple(t.to(dev, non_blocking=True) for t in v) if isinstance(v, tuple)
                       else v.to(dev, non_blocking=True)) for k, v in b.items()}
print('loaders ready (full-EHR dataset)')

## C3 · Backbone đông cứng + `forward_shared` / `forward_tail`

Chỉ còn vị trí LATE ⇒ `SHARED_UPTO = min(LATE) = 9`: blocks 0–8 chạy **một lần**
`no_grad` dùng chung cho cả 5 arm, mỗi arm chỉ chạy block 9–11. Kết quả numerically
y hệt 11 (block 4–8 ở 11 cũng `no_grad`, không tham số học nào phụ thuộc), chỉ nhanh hơn.

In [ ]:
from transformers import AutoModel
backbone = AutoModel.from_pretrained('microsoft/rad-dino').to(dev).eval()
assert os.path.exists(BB_CKPT), f'04b backbone khong thay: {BB_CKPT}'
_sd = torch.load(BB_CKPT, map_location=dev, weights_only=False)['model']
_bb = {k[9:]: v for k,v in _sd.items() if k.startswith('backbone.')}
assert _bb, 'no backbone.* keys in 04b checkpoint'
backbone.load_state_dict(_bb, strict=True)
for p in backbone.parameters(): p.requires_grad = False
LAYERS = backbone.encoder.layer; n_blocks = len(LAYERS); D = backbone.config.hidden_size
assert D == 768 and max(LATE) < n_blocks
SHARED_UPTO = min(LATE)
print(f'blocks {n_blocks} | shared no_grad 0..{SHARED_UPTO-1} | adapters LATE {LATE}')

def _layer(h, i):
    o = LAYERS[i](h); return o[0] if isinstance(o,(tuple,list)) else o

def forward_shared(x):
    with torch.no_grad():
        h = backbone.embeddings(x)
        for i in range(SHARED_UPTO): h = _layer(h, i)
    return h.detach()

def forward_tail(h, adapter_blocks=(), adapters=None, e=None):
    first = min(adapter_blocks) if adapter_blocks else n_blocks
    for i in range(SHARED_UPTO, n_blocks):
        with (torch.no_grad() if i < first else contextlib.nullcontext()):
            h = _layer(h, i)
        if adapters is not None and str(i) in adapters:
            h = adapters[str(i)](h, e)
    return backbone.layernorm(h)

# G0: manual loop == stock forward
_x = torch.randn(2, 3, MODEL_SIZE, MODEL_SIZE, device=dev)
with torch.no_grad():
    ref = backbone(pixel_values=_x).last_hidden_state
    got = forward_tail(forward_shared(_x))
md_ = (ref-got).abs().max().item()
assert torch.allclose(ref, got, atol=1e-4), f'manual != stock (max {md_:.2e})'
print(f'G0 manual forward == backbone(x).last_hidden_state  max|diff| {md_:.2e}  OK')

## C4 · EhrEncoder + CondAdapter (verbatim 11·C4)

`W_up` zero-init ⇒ `ΔZ = 0` ⇒ identity tuyệt đối ở step 0.

In [ ]:
Act = nn.GELU

class EhrEncoder(nn.Module):
    def __init__(self, d_e=PREREG['d_e'], d_hidden=256, dropout=0.2):
        super().__init__(); T = set(MICRO_TIERS); self.T = T
        self.mods = nn.ModuleDict(); d = 0
        def bag(key, vsize, dim):
            self.mods[key] = nn.EmbeddingBag(vsize, dim, mode='sum'); return dim
        self.set_emb = nn.Embedding(3, 4); d += 4
        d += 2*len(LABS) + 2*len(VITALS) + len(COMO_COLS)
        self.n_vit, self.n_como = len(VITALS), len(COMO_COLS)
        if 1 in T:
            d += bag('t1_org', ORG_V, 16) + bag('t1_spec', SPEC_V, 8)
            self.stat_emb = nn.Embedding(4, 4); d += 4 + 1 + 2*len(REC1)
        if 2 in T: d += bag('t2_spec', SPEC_V, 8) + 3
        if 3 in T:
            d += bag('t3_org', ORG_V, 16); self.rec_emb = nn.Embedding(3, 4)
            d += 4 + 2*len(REC3) + 1
        self.in_dim = d
        self.mlp = nn.Sequential(nn.Linear(d, d_hidden), nn.LayerNorm(d_hidden), Act(),
                                 nn.Dropout(dropout),
                                 nn.Linear(d_hidden, d_e), nn.LayerNorm(d_e), Act())
    def _bag(self, k, b):
        flat, off = b[k]; m = self.mods[k]
        return m(flat, off)
    def forward(self, b):
        T = self.T
        p = [self.set_emb(b['set_id']), b['lab_val'], b['lab_mask']]
        if self.n_vit:  p += [b['vit_val'], b['vit_mask']]
        if self.n_como: p += [b['como']]
        if 1 in T: p += [self._bag('t1_org',b), self._bag('t1_spec',b), self.stat_emb(b['stat_id']),
                         b['t1_same_encounter'][:,None], b['rec1_v'], b['rec1_m']]
        if 2 in T: p += [self._bag('t2_spec',b), b['t2_pending'][:,None],
                         b['t2_time_imprecise'][:,None], b['t2_n_pending'][:,None]]
        if 3 in T: p += [self._bag('t3_org',b), self.rec_emb(b['rec_id']), b['rec3_v'], b['rec3_m'],
                         b['t3_n_specimens'][:,None]]
        return self.mlp(torch.cat(p, 1))

class CondAdapter(nn.Module):
    def __init__(self, d=768, r=PREREG['bottleneck'], d_e=PREREG['d_e']):
        super().__init__()
        self.ln=nn.LayerNorm(d); self.down=nn.Linear(d,r); self.gen=nn.Linear(d_e,2*r); self.up=nn.Linear(r,d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
        nn.init.zeros_(self.gen.weight); nn.init.zeros_(self.gen.bias)
    def forward(self, Z, e):
        u = self.down(self.ln(Z)); g,b = self.gen(e).chunk(2,-1)
        u = (1+g[:,None,:])*u + b[:,None,:]
        return Z + self.up(F.gelu(u))
print('EhrEncoder in_dim:', EhrEncoder().in_dim, '-> d_e', PREREG['d_e'],
      '| params:', sum(p.numel() for p in EhrEncoder().parameters()))
print('adapter params/block:', sum(p.numel() for p in CondAdapter().parameters()))

## C5 · ArmModel (`cond` @ LATE) + D0 anchor — warm-start H5

Mọi arm là **cùng một** kiến trúc `L_E` của 11; khác nhau **chỉ** ở weight vector của loss.
`build_arm(SEED)` cho init xác định; `clone_arm` copy bit-for-bit để fork ở epoch 4.

In [ ]:
import sys, types
ML_SRC = '/content/ml_decoder.py'
if not os.path.exists(ML_SRC):
    !wget -q -O {ML_SRC} https://raw.githubusercontent.com/Alibaba-MIIL/ML_Decoder/main/src_files/ml_decoder/ml_decoder.py
sys.path.insert(0, '/content')
import ml_decoder as _mld
from ml_decoder import MLDecoder
if not hasattr(_mld.TransformerDecoderLayerOptimal, '_patched'):
    _oi=_mld.TransformerDecoderLayerOptimal.__init__
    def _init(self,*a,**kw): _oi(self,*a,**kw); self.self_attn=types.SimpleNamespace(batch_first=False)
    _mld.TransformerDecoderLayerOptimal.__init__=_init
    _of=_mld.TransformerDecoderLayerOptimal.forward
    def _fwd(self,tgt,memory,**kw): kw.pop('tgt_is_causal',None); kw.pop('memory_is_causal',None); return _of(self,tgt,memory,**kw)
    _mld.TransformerDecoderLayerOptimal.forward=_fwd
    _mld.TransformerDecoderLayerOptimal._patched=True

assert os.path.exists(H5_CKPT), f'H5 head khong thay: {H5_CKPT}'
H5_HEAD = torch.load(H5_CKPT, map_location=dev, weights_only=False)['model']
def _sub(prefix): return {k[len(prefix):]: v for k,v in H5_HEAD.items() if k.startswith(prefix)}
assert _sub('norm.') and _sub('mld.'), f'H5 checkpoint thieu norm.*/mld.* keys: {list(H5_HEAD)[:5]}'
print('H5 head <- token_fusion/checkpoints/H5_s42_best.pt  | norm', len(_sub('norm.')), 'mld', len(_sub('mld.')))

class ArmModel(nn.Module):
    """= arm L_E cua 11: EHR that -> CondAdapter @ block 10,11 (1-idx)."""
    def __init__(self, plain=False):
        super().__init__(); self.plain = plain; self.blocks = () if plain else tuple(LATE)
        self.norm = nn.LayerNorm(768); self.mld = MLDecoder(num_classes=len(LABELS),
                     initial_num_features=768, decoder_embedding=768, num_of_groups=len(LABELS))
        self.norm.load_state_dict(_sub('norm.')); self.mld.load_state_dict(_sub('mld.'))
        if not plain:
            self.enc = EhrEncoder()
            self.adapters = nn.ModuleDict({str(i): CondAdapter() for i in self.blocks})
    def forward(self, z_shared, batch):
        if self.plain: h = forward_tail(z_shared)
        else:          h = forward_tail(z_shared, self.blocks, self.adapters, self.enc(batch))
        return self.mld(self.norm(h[:, 1:]))

def build_arm(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    return ArmModel().to(dev)

def clone_arm(src):
    m = ArmModel().to(dev)
    m.load_state_dict({k: v.detach().clone() for k, v in src.state_dict().items()})
    return m

D0 = ArmModel(plain=True).to(dev).eval()
for p in D0.parameters(): p.requires_grad_(False)
dec_params = lambda m: list(m.norm.parameters()) + list(m.mld.parameters())
H5_DEC = [p.detach().clone() for p in dec_params(D0)]

_A0 = build_arm()
_n = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'trainable params/arm: {_n(_A0)/1e6:.3f}M'
      f'  | adapter {sum(p.numel() for p in _A0.adapters.parameters())}'
      f' | enc {sum(p.numel() for p in _A0.enc.parameters())}'
      f' | decoder {sum(p.numel() for p in dec_params(_A0))}')
_c = clone_arm(_A0)
assert all(torch.equal(a, b) for a, b in zip(_A0.state_dict().values(), _c.state_dict().values()))
print('clone_arm bit-for-bit OK | snapshot ~%.0f MB/arm' % (_n(_A0)*4/1e6))
del _c

## C6 · Cổng kiến trúc — identity step-0, EHR đọc labs+micro, L2-SP

In [ ]:
_b = next(iter(loader('val', eval_tf, False, 4, workers=0))); _b = to_dev(_b)
_zs = forward_shared(_b['image'])
_A0.eval(); D0.eval()
with torch.no_grad():
    y0 = D0(_zs, _b)
    d = (_A0(_zs, _b) - y0).abs().max().item()
    print(f'  L_E: |logit - D0| step-0 = {d:.2e}', 'OK' if d < 1e-4 else '!! KHONG identity')
    assert d < 1e-4
    e1 = _A0.enc(_b)
    _bl = dict(_b); _bl['lab_val'] = _b['lab_val'] + 3.0
    d_lab = (e1 - _A0.enc(_bl)).abs().mean().item()
    _cat, _off = _b['t1_org']; _bm = dict(_b)
    _bm['t1_org'] = (torch.full_like(_cat, (int(_cat.max())+1) % ORG_V), _off)
    d_mic = (e1 - _A0.enc(_bm)).abs().mean().item()
    print(f'  EHR nhay: labs |de| {d_lab:.3f} | micro t1_org |de| {d_mic:.3f}  (ca hai > 0)')
    assert d_lab > 1e-3 and d_mic > 1e-4

l2 = lambda m: sum(((p-ref)**2).sum() for p,ref in zip(dec_params(m), H5_DEC)).item()
l2_0 = l2(_A0)
_p = next(iter(_A0.norm.parameters())); _orig = _p.detach().clone()
with torch.no_grad(): _p.add_(0.01)
l2_1 = l2(_A0)
with torch.no_grad(): _p.copy_(_orig)
print(f'  L2-SP: init {l2_0:.2e} (=0) | sau khi nhich decoder {l2_1:.2e} (>0)')
assert l2_0 < 1e-8 and l2_1 > 0
print('GATES kien truc OK')

## C7 · Predict + pool 7:3 + **bộ metric đầy đủ** (§9)

`eval_all` trả về đúng 7 metric của bảng chính + mảng per-label để dựng §11.

- `AP` = `average_precision_score` (dùng chung cho mọi arm).
- `AUPRC` = `auc(recall, precision)` trên PR-curve — **tính riêng**, không thay bằng AP.
- `Recall@K_c`: xếp hạng mọi study theo score của label c, `K_c` = số dương của c trong
  val; `Recall@K_c = TP trong top-K_c / K_c`. Không dùng threshold.
- `Worst-3 AP` ở đây là per-arm (3 nhãn AP thấp nhất của chính arm đó); bảng delta ở C12
  dùng thêm phiên bản **tập nhãn cố định theo L0**.

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, auc

@torch.no_grad()
def predict(models, split, bs):
    for m in models.values(): m.eval()
    P = {t: np.zeros((len(IDX[split]), len(LABELS)), 'float32') for t in models}
    pos = 0
    for b in tqdm(loader(split, eval_tf, False, bs, workers=0), desc=f'pred[{split}]', leave=False):
        b = to_dev(b); z = forward_shared(b['image']); k = z.shape[0]
        for t, m in models.items(): P[t][pos:pos+k] = torch.sigmoid(m(z, b)).float().cpu().numpy()
        pos += k
    return P

def pool_study(p_img, split):
    sub = man.iloc[IDX[split]]; d = pd.DataFrame(p_img, columns=LABELS)
    d['study_id']=sub.study_id.to_numpy(); d['vg']=sub.view_group.to_numpy()
    sid = np.sort(d.study_id.unique())
    fr = d[d.vg=='frontal'].groupby('study_id')[LABELS].mean().reindex(sid)
    la = d[d.vg=='lateral'].groupby('study_id')[LABELS].mean().reindex(sid)
    hf = fr.notna().all(1).to_numpy().astype('float32'); hl = la.notna().all(1).to_numpy().astype('float32')
    den = (W_F*hf + W_L*hl)[:,None]
    return sid, (W_F*np.nan_to_num(fr.to_numpy('float32'))*hf[:,None]
                 + W_L*np.nan_to_num(la.to_numpy('float32'))*hl[:,None]) / den

def recall_at_k(y, s):
    K = int(y.sum())
    if K == 0: return np.nan
    order = np.argsort(-s, kind='stable')[:K]
    return float(y[order].sum()) / K

def eval_all(p_img, split='val'):
    sid, PS = pool_study(p_img, split)
    Y = ehr.set_index('study_id').loc[sid, LABELS].to_numpy('float32')
    C = len(LABELS)
    ap    = np.array([average_precision_score(Y[:,j], PS[:,j]) if Y[:,j].sum() else np.nan for j in range(C)])
    auprc = np.full(C, np.nan); auroc = np.full(C, np.nan); rk = np.full(C, np.nan)
    for j in range(C):
        if Y[:,j].sum() == 0 or Y[:,j].sum() == len(Y): continue
        pr, rc, _ = precision_recall_curve(Y[:,j], PS[:,j])
        auprc[j] = auc(rc, pr)
        auroc[j] = roc_auc_score(Y[:,j], PS[:,j])
        rk[j]    = recall_at_k(Y[:,j], PS[:,j])
    w3 = float(np.sort(ap[~np.isnan(ap)])[:3].mean())
    T = TAIL_IDX
    return dict(macro_map=float(np.nanmean(ap)),      tail_map=float(np.nanmean(ap[T])),
                head_map=float(np.nanmean(ap[TIER=='head'])), med_map=float(np.nanmean(ap[TIER=='medium'])),
                worst3_ap=w3,
                macro_auprc=float(np.nanmean(auprc)), macro_auroc=float(np.nanmean(auroc)),
                tail_auroc=float(np.nanmean(auroc[T])), tail_recall_k=float(np.nanmean(rk[T])),
                n_val_studies=int(len(sid)),
                per_ap=ap.tolist(), per_auprc=auprc.tolist(), per_auroc=auroc.tolist(), per_rk=rk.tolist())

METRIC_KEYS = ['macro_map','tail_map','worst3_ap','macro_auprc','macro_auroc','tail_auroc','tail_recall_k']
print('predict / pool 7:3 / eval_all ready |', len(METRIC_KEYS), 'metric chinh')

# ---- §10: Worst-3 la TAP NHAN KHOA, khong suy lai tu arm DBL ----
assert all(l in LABELS for l in WORST3_FIXED), f'{WORST3_FIXED} khong nam trong LABELS: {LABELS}'
W3_IDX = np.array([LABELS.index(l) for l in WORST3_FIXED])
w3_fixed = lambda r: float(np.nanmean(np.array(r['per_ap'])[W3_IDX]))
print('Worst-3 KHOA (§10):', [f'{l}(idx {i}, tier {TIER[i]}, n={int(N_POS_TRAIN[i]):,})'
                              for l, i in zip(WORST3_FIXED, W3_IDX)])

## C8 · Audit bắt buộc cấu hình DB **cũ** (§5)

Lấy từ code thật, không từ trí nhớ: [`04b_raddino_lpft.ipynb`] cell 8 (`class ResampleLoss`)
và cell 14 (call site `ResampleLoss(class_freq=cf['class_freq'], train_num=cf['num_train'])`
— **tất cả tham số để mặc định**). 04b chính là notebook sinh ra `raddino_mld_db_ft_best.pt`,
backbone mà nb11/nb12/nb13 đang dùng.

Ba chỗ legacy lệch khỏi config COCO chính thức — cần biết trước khi diễn giải kết quả cũ.

In [ ]:
LEGACY_DB = dict(
    source='04b_raddino_lpft.ipynb cell 8 (class ResampleLoss) + cell 14 (call site, all defaults)',
    use_sigmoid=True, reduction='mean', loss_weight=1.0,
    reweight_func='rebalance',
    focal=dict(focal=True, alpha=0.5, gamma=2.0),
    map_param=dict(alpha=0.1, beta=10.0, gamma=0.9),          # gamma = paper mu (KHONG phai focal gamma)
    logit_reg=dict(neg_scale=2.0, init_bias=0.05),
    class_freq="class_freq.pkl['class_freq'] = study-level TRAIN positives",
    train_num="class_freq.pkl['num_train']",
    sampler='natural (04b khong dung class-aware sampler)',
)
OFFICIAL_COCO = dict(
    reweight_func='rebalance',
    focal=dict(focal=True, balance_param=2.0, gamma=2.0),
    map_param=dict(alpha=0.1, beta=10.0, gamma=0.2),
    logit_reg=dict(neg_scale=2.0, init_bias=0.05),
    sampler='class-aware sampler',
)
print('=== LEGACY DB CONFIG (audit §5) ===')
for k, v in LEGACY_DB.items(): print(f'  {k:16s} {v}')
print('\n=== vs official COCO DB config ===')
for k, v in OFFICIAL_COCO.items(): print(f'  {k:16s} {v}')

print(f'''
=== BA KHAC BIET LEGACY vs OFFICIAL (quan trong khi doc lai ket qua "ASL > DB Loss") ===

1. map_param['gamma'] = mu:  legacy {LEGACY_DB['map_param']['gamma']}  vs  official {OFFICIAL_COCO['map_param']['gamma']}
   mu la vi tri chuyen tiep cua sigmoid tren thang r. HUONG tac dong cua mu PHU THUOC
   phan phoi r cua chinh cohort nay -> KHONG suy dien, C10 do truc tiep tren 147,791
   study train. Day la thu screen nay tune (D1/D2/D3 = .10/.20/.30).

2. He so focal:  legacy alpha_t = where(y==1, 0.5, 1-0.5) = 0.5 HANG SO
   official       balance_param = 2.0 nhan thang.
   -> loss scale legacy = 0.25x official. Voi reduction='mean' + LR co dinh, day thuc chat
      la khac LEARNING RATE, khong phai khac thiet ke loss.

3. Sampler: official dung class-aware sampler; legacy va screen nay dung NATURAL (§5 co y giu,
   doi sampler + loss cung luc se confound).

LUU Y (§8): plan gia dinh mu thap = re-balance manh hon, dua tren gia tri paper (COCO .2 / VOC .3).
   Gia dinh do dung cho phan phoi r cua COCO/VOC, CHUA chac dung o cohort 13-nhan nay.
   C10 do 2 ty le -> doc o do truoc khi dien giai ket qua.

=> Ket luan cu "ASL > DB Loss" chi valid duoi cau hinh nay, dung suy rong.
   §7 yeu cau GIU nguyen ho DB-Focal + focal params cho MOI arm -> alpha=0.5, gamma=2.0
   co dinh, chi mu thay doi. D0 giu y het legacy de tra loi "config cu con te khong?".''')

DB_FIXED = dict(reweight_func=LEGACY_DB['reweight_func'], focal=dict(LEGACY_DB['focal']),
                map_alpha=LEGACY_DB['map_param']['alpha'], map_beta=LEGACY_DB['map_param']['beta'],
                neg_scale=LEGACY_DB['logit_reg']['neg_scale'],
                init_bias=LEGACY_DB['logit_reg']['init_bias'],
                loss_weight=LEGACY_DB['loss_weight'], reduction='mean')
assert DB_MU['D0'] == LEGACY_DB['map_param']['gamma'], 'D0 phai la mu cua legacy'
_dup = [a for a in ('D1','D2','D3') if DB_MU[a] == DB_MU['D0']]
print(f'\nD0 mu = {DB_MU["D0"]} (legacy) | D1-D3 mu = '
      f'{[DB_MU[a] for a in ("D1","D2","D3")]} | trung voi D0: {_dup or "khong"}')
assert not _dup, f'{_dup} trung D0 -> bo bot arm cho do phi GPU (§7)'
json.dump(dict(legacy=LEGACY_DB, official_coco=OFFICIAL_COCO, db_fixed=DB_FIXED, db_mu=DB_MU),
          open(f'{SC_DIR}/metrics/db_config_audit.json','w'), indent=1)
print('-> metrics/db_config_audit.json')

## C9 · DB-Loss implementation + Stage 0 (§6.2)

`ResampleLoss` dán **verbatim** từ 04b cell 8 (công thức không sửa một ký tự) — đây là
"legacy code" trong bài test tương đương. `db_reference` là bản viết lại **độc lập** từ công
thức trong paper, dùng để bắt lỗi port. Hai bên phải khớp `loss < 1e-7`, `grad < 1e-6`.

`DBArm` / `ASLArm` chỉ là lớp bọc cho giao diện chung `(loss, pos_per_class, neg_per_class)`
để log diagnostics — `DBArm` gọi đúng `ResampleLoss(reduction='none')` rồi `.mean()`, phải
bằng đúng `reduction='mean'`.

In [ ]:
# ============ VERBATIM tu 04b_raddino_lpft.ipynb cell 8 — KHONG SUA CONG THUC ============
class ResampleLoss(nn.Module):
    """DB Loss (Wu et al., ECCV 2020). Ported from wutong16/DistributionBalancedLoss."""
    def __init__(self, class_freq, train_num,
                 use_sigmoid=True, reduction='mean', loss_weight=1.0,
                 reweight_func='rebalance',
                 focal=dict(focal=True, alpha=0.5, gamma=2.0),
                 map_param=dict(alpha=0.1, beta=10.0, gamma=0.9),
                 logit_reg=dict(neg_scale=2.0, init_bias=0.05)):
        super().__init__()
        assert use_sigmoid
        self.reduction, self.loss_weight, self.reweight_func = reduction, loss_weight, reweight_func
        self.focal    = focal['focal']; self.gamma = focal['gamma']; self.alpha = focal['alpha']
        self.map_alpha, self.map_beta, self.map_gamma = \
            map_param['alpha'], map_param['beta'], map_param['gamma']

        cf = torch.as_tensor(np.asarray(class_freq), dtype=torch.float)
        self.register_buffer('class_freq', cf)
        self.train_num   = float(train_num)
        self.register_buffer('freq_inv',      torch.ones_like(cf) / cf)
        self.register_buffer('propotion_inv', self.train_num / cf)

        self.neg_scale = logit_reg.get('neg_scale', 1.0)
        init_bias      = logit_reg.get('init_bias', 0.0)
        self.register_buffer('init_bias',
            -torch.log(self.train_num / cf - 1) * init_bias / self.neg_scale)
        self.logit_reg = logit_reg

    def rebalance_weight(self, gt_labels):
        repeat_rate = torch.sum(gt_labels * self.freq_inv, dim=1, keepdim=True)
        pos_weight  = self.freq_inv.unsqueeze(0) / repeat_rate
        return torch.sigmoid(self.map_beta * (pos_weight - self.map_gamma)) + self.map_alpha

    def logit_reg_functions(self, labels, logits, weight):
        if not self.logit_reg: return logits, weight
        if 'init_bias' in self.logit_reg:
            logits = logits + self.init_bias
        if 'neg_scale' in self.logit_reg:
            logits = logits * (1 - labels) * self.neg_scale + logits * labels
            weight = weight / self.neg_scale * (1 - labels) + weight * labels
        return logits, weight

    def forward(self, cls_score, label):
        label  = label.float()
        weight = self.rebalance_weight(label) if self.reweight_func == 'rebalance' \
                 else torch.ones_like(label)
        cls_score, weight = self.logit_reg_functions(label, cls_score, weight)
        if self.focal:
            logpt = F.binary_cross_entropy_with_logits(cls_score, label, reduction='none')
            pt    = torch.exp(-logpt)
            wtloss  = F.binary_cross_entropy_with_logits(cls_score, label,
                                                         weight=weight, reduction='none')
            alpha_t = torch.where(label == 1, self.alpha, 1 - self.alpha)
            loss = alpha_t * ((1 - pt) ** self.gamma) * wtloss
        else:
            loss = F.binary_cross_entropy_with_logits(cls_score, label,
                                                      weight=weight, reduction='none')
        if self.reduction == 'mean': loss = loss.mean()
        elif self.reduction == 'sum': loss = loss.sum()
        return self.loss_weight * loss
# =========================================================================================

def db_reference(logits, label, cf, train_num, mu, map_a=0.1, map_b=10.0,
                 neg_scale=2.0, init_bias=0.05, f_alpha=0.5, f_gamma=2.0):
    """Viet lai DOC LAP tu cong thuc paper (§6.2) de bat loi port. Khong dung ResampleLoss."""
    y  = label.float()
    fi = 1.0 / torch.as_tensor(np.asarray(cf), dtype=torch.float, device=logits.device)
    r  = fi[None, :] / (y * fi[None, :]).sum(1, keepdim=True)          # r_{k,i}
    w  = torch.sigmoid(map_b * (r - mu)) + map_a                       # r_hat
    nu = -torch.log(torch.as_tensor(float(train_num), device=logits.device) /
                    torch.as_tensor(np.asarray(cf), dtype=torch.float, device=logits.device) - 1
                    ) * init_bias / neg_scale
    z  = logits + nu[None, :]
    z  = torch.where(y == 1, z, z * neg_scale)                         # NTR tren logit
    w  = torch.where(y == 1, w, w / neg_scale)                         # ... va tren weight
    lp, l1mp = F.logsigmoid(z), F.logsigmoid(-z)       # on dinh so hoc o |z| lon
    logpt = y * lp + (1-y) * l1mp
    bce, pt = -logpt, torch.exp(logpt)
    at  = torch.where(y == 1, torch.full_like(y, f_alpha), torch.full_like(y, 1 - f_alpha))
    return (at * (1 - pt) ** f_gamma * (w * bce)).mean()

class DBArm(nn.Module):
    """Bo ResampleLoss -> giao dien chung (loss, pos_per_class, neg_per_class)."""
    def __init__(s, mu, class_freq, train_num, **fixed):
        super().__init__()
        s.db = ResampleLoss(class_freq=class_freq, train_num=train_num, reduction='none',
                            loss_weight=fixed['loss_weight'], reweight_func=fixed['reweight_func'],
                            focal=fixed['focal'],
                            map_param=dict(alpha=fixed['map_alpha'], beta=fixed['map_beta'], gamma=mu),
                            logit_reg=dict(neg_scale=fixed['neg_scale'], init_bias=fixed['init_bias']))
        s.mu = mu
    def forward(s, x, y, parts=False):
        el = s.db(x, y); loss = el.mean()
        if parts:
            yb = y.float()
            return loss, (el*yb).mean(0).detach(), (el*(1-yb)).mean(0).detach()
        return loss

class ASLArm(nn.Module):
    """ASL cua nb11/nb12 (gp=1, gn=4, clip=.05) — cung giao dien."""
    def __init__(s, gn=4, gp=1, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y, parts=False):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lp = y*torch.log(xp.clamp(min=s.eps)); ln = (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y); mod = torch.pow(1-pt, g)
        Lp = -lp*mod; Ln = -ln*mod; loss = (Lp+Ln).mean()
        if parts: return loss, Lp.mean(0).detach(), Ln.mean(0).detach()
        return loss

def _asl_ref(x, y, gn=4, gp=1, clip=0.05, eps=1e-8):
    y=y.float(); xp=torch.sigmoid(x); xn=1-xp
    if clip: xn=(xn+clip).clamp(max=1)
    los=y*torch.log(xp.clamp(min=eps))+(1-y)*torch.log(xn.clamp(min=eps))
    pt=xp*y+xn*(1-y); g=gp*y+gn*(1-y); return -(los*torch.pow(1-pt,g)).mean()

CF, NTR_N = N_POS_TRAIN.tolist(), N_TRAIN
torch.manual_seed(0)
_x = (torch.randn(96, len(LABELS), device=dev) * 3).requires_grad_(True)
_y = torch.zeros(96, len(LABELS), device=dev)
for i in range(96):                                   # 1-4 nhan duong/sample, giong that
    _y[i, torch.randperm(len(LABELS), device=dev)[:int(torch.randint(1, 5, (1,)))]] = 1.

print('=== §6.2 legacy vs reference implementation ===')
for _mu in [LEGACY_DB['map_param']['gamma']] + [DB_MU[a] for a in ('D1','D2','D3')]:
    leg = ResampleLoss(class_freq=CF, train_num=NTR_N, reduction='mean',
                       map_param=dict(alpha=0.1, beta=10.0, gamma=_mu)).to(dev)
    l_leg = leg(_x, _y); l_ref = db_reference(_x, _y, CF, NTR_N, _mu)
    g_leg = torch.autograd.grad(l_leg, _x, retain_graph=True)[0]
    g_ref = torch.autograd.grad(l_ref, _x, retain_graph=True)[0]
    dl = abs(l_leg.item()-l_ref.item()); dg = (g_leg-g_ref).abs().max().item()
    print(f'  mu={_mu:<5} loss legacy {l_leg.item():.10f} ref {l_ref.item():.10f} | '
          f'|dloss| {dl:.2e} (<1e-7) | max|dgrad| {dg:.2e} (<1e-6)',
          'OK' if dl < 1e-7 and dg < 1e-6 else '!! FAIL')
    assert dl < 1e-7 and dg < 1e-6

print('\n=== DBArm(reduction=none).mean() == ResampleLoss(reduction=mean) ===')
for a in ('D0','D1','D2','D3'):
    arm = DBArm(DB_MU[a], CF, NTR_N, **DB_FIXED).to(dev)
    ref = ResampleLoss(class_freq=CF, train_num=NTR_N, reduction='mean', loss_weight=DB_FIXED['loss_weight'],
                       reweight_func=DB_FIXED['reweight_func'], focal=DB_FIXED['focal'],
                       map_param=dict(alpha=DB_FIXED['map_alpha'], beta=DB_FIXED['map_beta'], gamma=DB_MU[a]),
                       logit_reg=dict(neg_scale=DB_FIXED['neg_scale'], init_bias=DB_FIXED['init_bias'])).to(dev)
    d = abs(arm(_x, _y).item() - ref(_x, _y).item())
    print(f'  {a} mu={DB_MU[a]:<5} |diff| {d:.2e}', 'OK' if d < 1e-9 else '!! FAIL'); assert d < 1e-9

print('\n=== ASLArm == ASL cua nb11/nb12 (bit-for-bit) ===')
_la, _lr = ASLArm().to(dev)(_x, _y), _asl_ref(_x, _y)
print(f'  ASLArm {_la.item():.10f} | nb12 ASL {_lr.item():.10f} | |diff| {abs(_la.item()-_lr.item()):.1e}',
      'OK' if torch.equal(_la, _lr) else '!! FAIL')
assert torch.equal(_la, _lr)

print('\n=== sanity khac ===')
_leg0 = ResampleLoss(class_freq=CF, train_num=NTR_N).to(dev)
_zero = torch.zeros(4, len(LABELS), device=dev)                    # sample KHONG co nhan duong
_l0 = _leg0(torch.randn(4, len(LABELS), device=dev), _zero)
print(f'  sample toan-am (repeat_rate=0 -> pos_weight=inf): loss {_l0.item():.6f} finite',
      'OK' if torch.isfinite(_l0) else '!! NaN/inf'); assert torch.isfinite(_l0)
_rare, _head = int(np.argmin(N_POS_TRAIN)), int(np.argmax(N_POS_TRAIN))
_anchor = torch.zeros(4, len(LABELS), device=dev); _anchor[:, _head] = 1.   # neo o class pho bien nhat
for a in ('D0','D2'):
    _w = ResampleLoss(class_freq=CF, train_num=NTR_N,
                      map_param=dict(alpha=0.1, beta=10.0, gamma=DB_MU[a])).to(dev)
    _wm = _w.rebalance_weight(_anchor).mean(0).cpu().numpy()
    print(f'  {a} mu={DB_MU[a]}: trong sample chi duong "{LABELS[_head]}", '
          f'w[{LABELS[_rare]}]={_wm[_rare]:.4f} > w[{LABELS[_head]}]={_wm[_head]:.4f}',
          'OK' if _wm[_rare] > _wm[_head] else '!! sai huong'); assert _wm[_rare] > _wm[_head]
# tinh chat cau truc: sample CHI co 1 nhan duong -> r=1 cho chinh nhan do, bat ke tan suat
_eye = torch.eye(len(LABELS), device=dev)
_diag = ResampleLoss(class_freq=CF, train_num=NTR_N,
                     map_param=dict(alpha=0.1, beta=10.0, gamma=0.2)).to(dev
                     ).rebalance_weight(_eye).diagonal().cpu().numpy()
print(f'  sample don-nhan: w cua chinh nhan do = {_diag.min():.6f}..{_diag.max():.6f} (HANG SO theo cau truc,'
      ' vi r = freq_inv_c/freq_inv_c = 1)')
assert np.allclose(_diag, _diag[0]), 'ky vong hang so'
print('  => class-awareness cua DBL chi phat huy qua DONG XUAT HIEN nhan, khong qua tan suat don le.')
print('\nSTAGE 0 PASS')

## C10 · Phân phối DB weight (§6.3)

Khác WASL, weight của DBL **không phải một scalar cố định mỗi class**: nó phụ thuộc cả tần
suất class lẫn **các nhãn dương đồng xuất hiện** trong chính study đó. Nên báo cáo phân phối,
không phải bảng 13 số.

Tính trên **toàn bộ 147,791 study TRAIN** (chỉ cần vector nhãn, không cần model) nên rẻ.
Báo cáo cả `r_hat` thô lẫn weight hiệu dụng sau NTR (`w/neg_scale` ở ô âm) — cái sau mới là
số thật sự nhân vào BCE.

In [ ]:
Y_TR = ehr.loc[(ehr.split=='train').to_numpy(), LABELS].to_numpy('float32')
assert Y_TR.shape[0] == N_TRAIN
_Yt = torch.from_numpy(Y_TR).to(dev)

def db_weight_stats(mu, chunk=20000):
    L = ResampleLoss(class_freq=CF, train_num=NTR_N,
                     map_param=dict(alpha=DB_FIXED['map_alpha'], beta=DB_FIXED['map_beta'], gamma=mu),
                     logit_reg=dict(neg_scale=DB_FIXED['neg_scale'], init_bias=DB_FIXED['init_bias'])).to(dev)
    W = torch.cat([L.rebalance_weight(_Yt[i:i+chunk]) for i in range(0, len(_Yt), chunk)]).cpu().numpy()
    return W

def qrow(name, v):
    v = v[np.isfinite(v)]
    return dict(scope=name, n=int(v.size), min=float(v.min()), p05=float(np.percentile(v,5)),
                median=float(np.median(v)), mean=float(v.mean()),
                p95=float(np.percentile(v,95)), max=float(v.max()))

WSTAT, WPOS = {}, {}
for a in ('D0','D1','D2','D3'):
    W = db_weight_stats(DB_MU[a]); WSTAT[a] = W
    rows = [qrow('overall', W.ravel())]
    for t in ('head','medium','tail'): rows.append(qrow(f'tier:{t}', W[:, TIER==t].ravel()))
    for l, j in zip(WORST3_FIXED, W3_IDX): rows.append(qrow(f'worst3:{l}', W[:, j]))
    df = pd.DataFrame(rows)
    print(f'\n===== {ARM_LABEL[a]} (mu={DB_MU[a]}) — r_hat = sigmoid(beta*(r-mu)) + alpha =====')
    print(df.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    # mean POSITIVE db weight per class (§6.3)
    mp = np.array([W[Y_TR[:,j]==1, j].mean() if Y_TR[:,j].sum() else np.nan for j in range(len(LABELS))])
    WPOS[a] = mp
    df.to_csv(f'{SC_DIR}/metrics/db_weight_dist_{a}.csv', index=False)

pos_tab = pd.DataFrame({'Label': LABELS, 'N positive': N_POS_TRAIN.astype(int), 'Tier': TIER,
                        **{ARM_LABEL[a]: WPOS[a] for a in ('D0','D1','D2','D3')}}
                      ).sort_values('N positive', ascending=False).reset_index(drop=True)
print('\n===== mean POSITIVE DB weight moi class (§6.3) =====')
print(pos_tab.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
pos_tab.to_csv(f'{SC_DIR}/metrics/db_weight_pos_per_class.csv', index=False)

print('''
=== mu dieu khien DONG THOI hai thu — phai doc ca hai ===

  (a) tail/head : phan biet giua class hiem va class pho bien, TRONG cac o duong
  (b) pos/neg   : can bang tong the giua o duong va o am (sau NTR: w_neg = r_hat/neg_scale)

Voi o DUONG, r = freq_inv_k / sum_j(y_j*freq_inv_j) <= 1 luon dung. Voi o AM, r thuong >> 1
(class hiem khong duong trong mot study toan nhan pho bien) -> sigmoid bao hoa ~1 bat ke mu.
=> mu chi dich chuyen vung chuyen tiep tren phia DUONG.''')
_rows = []
for a in ('D0','D1','D2','D3'):
    W = WSTAT[a]
    wpos = np.array([W[Y_TR[:,j]==1, j].mean() for j in range(len(LABELS))])
    wneg = np.array([W[Y_TR[:,j]==0, j].mean() for j in range(len(LABELS))]) / DB_FIXED['neg_scale']
    _rows.append(dict(arm=ARM_LABEL[a], mu=DB_MU[a],
                      w_tail=np.nanmean(wpos[TIER=='tail']), w_head=np.nanmean(wpos[TIER=='head']),
                      tail_head=np.nanmean(wpos[TIER=='tail'])/np.nanmean(wpos[TIER=='head']),
                      w_pos=np.nanmean(wpos), w_neg_eff=np.nanmean(wneg),
                      pos_neg=np.nanmean(wpos)/np.nanmean(wneg)))
rt = pd.DataFrame(_rows)
print(rt.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print(f'  de so — WASL .50 (nb12): tail/head = {2.4636/0.5701:.3f} (tail 2.4636 / head 0.5701), '
      'pos/neg = 1.000 (ASL khong dung weight o o am)')
rt.to_csv(f'{SC_DIR}/metrics/db_weight_ratios.csv', index=False)
_best = rt.loc[rt.tail_head.idxmax()]
print(f'''
Doc: arm co tail/head cao nhat la {_best.arm} (mu={_best.mu}). Neu do la D0 (legacy) thi
grid mu = .10/.20/.30 cua plan chi kham pha huong re-balance YEU HON legacy, va ket qua
"DBL khong an" se KHONG duoc dien giai la "da tune du manh". Ghi ro dieu nay vao bao cao.''')
del _Yt; torch.cuda.empty_cache()

## C11 · Train helpers (verbatim nb12·C10) + probe batch

Không đổi gì: optimizer 2 LR-group, cosine + warmup, delayed unfreeze, L2-SP, clip riêng
module, `sync_rng` để mọi arm dùng chung dropout mask, `grad_norms` đo trước clip.

In [ ]:
def make_opt(m):
    ad = list(m.adapters.parameters()) + list(m.enc.parameters())
    de = dec_params(m)
    def split(ps, lr):
        dec=[p for p in ps if p.ndim>1]; nod=[p for p in ps if p.ndim<=1]
        return [{'params':dec,'lr':lr,'weight_decay':PREREG['wd']},
                {'params':nod,'lr':lr,'weight_decay':0.0}]
    return torch.optim.AdamW(split(ad, LR_ADAPTER) + split(de, LR_DECODER))

def make_sched(opt, total, warm):
    cw = lambda s: s/max(warm,1) if s<warm else 0.5*(1+math.cos(math.pi*(s-warm)/max(total-warm,1)))
    const = lambda s: 1.0
    return torch.optim.lr_scheduler.LambdaLR(opt, [cw, cw, const, const])

def set_decoder_trainable(m, on):
    for p in dec_params(m): p.requires_grad_(on)

def l2sp_term(m):
    return sum(((p-ref)**2).sum() for p,ref in zip(dec_params(m), H5_DEC))

def _gnorm(ps):
    gs = [p.grad.detach() for p in ps if p.grad is not None]
    if not gs: return torch.zeros((), device=dev)
    return torch.linalg.vector_norm(torch.stack([torch.linalg.vector_norm(g) for g in gs]))

def grad_norms(m):
    return (_gnorm(dec_params(m)), _gnorm(list(m.enc.parameters())), _gnorm(list(m.adapters.parameters())))

def clip_arm(m):
    dec = [p for p in dec_params(m) if p.grad is not None]
    oth = [p for p in (list(m.adapters.parameters())+list(m.enc.parameters())) if p.grad is not None]
    if dec: nn.utils.clip_grad_norm_(dec, CLIP_DECODER)
    if oth: nn.utils.clip_grad_norm_(oth, CLIP_OTHER)

def sync_rng(ep, i):
    s = (SEED*1_000_003 + ep*100_003 + i) % (2**31-1)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def probe_batch(cands=(16,32,48,64,96)):
    if dev!='cuda': return 8
    A = [build_arm() for _ in range(len(ARMS))]; crit = DBArm(0.2, CF, NTR_N, **DB_FIXED).to(dev)
    for m in A: set_decoder_trainable(m, True)
    ok = cands[0]
    for B in cands:
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            bb = to_dev(collate([ImgDS(IDX['train'],train_tf)[k] for k in range(B)]))
            z = forward_shared(bb['image'])
            for m in A:
                (crit(m(z,bb),bb['y']) + L2SP_DECODER*l2sp_term(m)).backward(); m.zero_grad(set_to_none=True)
            pk = torch.cuda.max_memory_allocated()/1e9; tot = torch.cuda.get_device_properties(0).total_memory/1e9
            print(f'  B={B:3d} peak {pk:5.1f}/{tot:.0f} GB ({pk/tot:5.1%})')
            ok = B if pk/tot < 0.80 else ok
            if pk/tot >= 0.80: break
        except torch.cuda.OutOfMemoryError:
            print(f'  B={B} OOM'); break
    del A; torch.cuda.empty_cache(); return ok

BATCH = probe_batch(); print('-> BATCH', BATCH, '| accum', max(1, EFF//BATCH), '-> effective', EFF)

## C12 · Main run — 5 arm × 8 epoch, không fork

Cả 5 arm khởi tạo **bit-for-bit giống nhau** (`clone_arm` từ một `build_arm(42)`), cùng thứ tự
batch, cùng dropout mask mỗi step. Khác biệt duy nhất là hàm loss.

Ghi `_resume.pt` + snapshot mỗi epoch → chạy lại cell này là tự resume sau khi Colab rớt.

In [ ]:
LOG_PATH = f'{SC_DIR}/metrics/epoch_log.json'
snap_path   = lambda arm, ep: f'{SNAP_DIR}/{arm}_ep{ep}.pt'
resume_path = lambda: f'{SC_DIR}/checkpoints/_resume.pt'
C_LAB = len(LABELS)

def make_crit():
    c = {'A0': ASLArm().to(dev)}
    for a in ('D0','D1','D2','D3'): c[a] = DBArm(DB_MU[a], CF, NTR_N, **DB_FIXED).to(dev)
    return c

def train_all(batch=BATCH, epochs=EPOCHS):
    dl    = loader('train', train_tf, True, batch)
    accum = max(1, EFF//batch)
    TOTAL = epochs*math.ceil(len(dl)/accum); WARM = int(PREREG['warmup_frac']*TOTAL)
    CRIT  = make_crit()
    _mk = lambda m: (lambda o: dict(m=m, opt=o, sch=make_sched(o, TOTAL, WARM)))(make_opt(m))

    LOG = json.load(open(LOG_PATH)) if os.path.exists(LOG_PATH) else {}
    st, start = {}, 0
    if os.path.exists(resume_path()):
        c = torch.load(resume_path(), map_location=dev, weights_only=False)
        if c.get('batch') != batch:
            raise RuntimeError(f"resume batch {c.get('batch')} != {batch}; xoa {resume_path()}")
        for t in ARMS:
            m = ArmModel().to(dev); m.load_state_dict(c[t]['model'])
            o = make_opt(m); o.load_state_dict(c[t]['opt'])
            s = make_sched(o, TOTAL, WARM); s.load_state_dict(c[t]['sch'])
            st[t] = dict(m=m, opt=o, sch=s)
        start = c['epoch'] + 1
        print(f'RESUMED @ep{start+1}')
        if start >= epochs: print('da chay xong 8 epoch -> sang C13'); return LOG
    else:
        base = build_arm(SEED)
        for t in ARMS: st[t] = _mk(base if t == ARMS[0] else clone_arm(base))
        _sd = [st[t]['m'].state_dict() for t in ARMS]
        assert all(torch.equal(a, b) for d in _sd[1:] for a, b in zip(_sd[0].values(), d.values())), \
            'cac arm KHONG cung init'
        print(f'init: {len(ARMS)} arm cung init bit-for-bit | steps/epoch {math.ceil(len(dl)/accum)} '
              f'| total {TOTAL} | warmup {WARM}')

    for ep in range(start, epochs):
        unfrozen = ep >= DECODER_WARMUP_EPOCHS
        tag = '[dec]' if unfrozen else '[warm]'
        for t in ARMS: set_decoder_trainable(st[t]['m'], unfrozen); st[t]['m'].train()
        z0 = lambda: torch.zeros(C_LAB, device=dev)
        dg = {t: dict(loss=torch.zeros((), device=dev), n=0, pos=z0(), neg=z0(),
                      gd=torch.zeros((), device=dev), ge=torch.zeros((), device=dev),
                      ga=torch.zeros((), device=dev), nopt=0) for t in ARMS}

        torch.manual_seed(SEED*7919 + ep)                     # thu tu batch on dinh qua resume
        for i, b in enumerate(tqdm(dl, desc=f'ep{ep+1}/{epochs}{tag}', leave=False)):
            b = to_dev(b); z = forward_shared(b['image'])
            step_now = ((i+1) % accum == 0) or (i == len(dl)-1)
            for t in ARMS:
                e = st[t]; d = dg[t]
                sync_rng(ep, i)
                loss, lp, ln = CRIT[t](e['m'](z, b), b['y'], parts=True)
                full = loss + L2SP_DECODER*l2sp_term(e['m']) if unfrozen else loss
                (full/accum).backward()
                d['loss'] += loss.detach(); d['pos'] += lp; d['neg'] += ln; d['n'] += 1
                if step_now:
                    gd, ge, ga = grad_norms(e['m'])
                    d['gd'] += gd; d['ge'] += ge; d['ga'] += ga; d['nopt'] += 1
                    clip_arm(e['m']); e['opt'].step(); e['sch'].step(); e['opt'].zero_grad(set_to_none=True)

        Pv = predict({t: st[t]['m'] for t in ARMS}, 'val', batch*2)
        rows = []
        for t in ARMS:
            r = eval_all(Pv[t], 'val'); r['worst3_fixed'] = w3_fixed(r)
            d = dg[t]; n = max(d['n'],1); no = max(d['nopt'],1)
            pos = (d['pos']/n).cpu().numpy(); neg = (d['neg']/n).cpu().numpy()
            tm = lambda v: {k: float(v[TIER==k].mean()) for k in ('head','medium','tail')}
            r['diag'] = dict(train_loss=float(d['loss'].item()/n), pos=tm(pos), neg=tm(neg),
                             grad_decoder=float(d['gd'].item()/no), grad_enc=float(d['ge'].item()/no),
                             grad_adapter=float(d['ga'].item()/no),
                             lr_adapter=st[t]['opt'].param_groups[0]['lr'], unfrozen=bool(unfrozen))
            LOG.setdefault(t, {})[str(ep+1)] = r
            torch.save({k: v.detach().cpu() for k, v in st[t]['m'].state_dict().items()}, snap_path(t, ep+1))
            rows.append(f"{t} mAP {r['macro_map']:.4f} tail {r['tail_map']:.4f} "
                        f"w3f {r['worst3_fixed']:.4f} rk {r['tail_recall_k']:.3f} L {r['diag']['train_loss']:.4f}")
        json.dump(LOG, open(LOG_PATH,'w'), indent=1)
        torch.save({**{t: dict(model=st[t]['m'].state_dict(), opt=st[t]['opt'].state_dict(),
                               sch=st[t]['sch'].state_dict()) for t in ARMS},
                    'epoch': ep, 'batch': batch}, resume_path()+'.tmp')
        os.replace(resume_path()+'.tmp', resume_path())
        print(f'  ep{ep+1}{tag} | ' + ' | '.join(rows) + '  [saved]')
    print('\nDONE 8 epoch x', len(ARMS), 'arm ->', LOG_PATH)
    return LOG

LOG = train_all()

## C13 · Chọn checkpoint (§11) + bảng chính + so với ASL/WASL (§13, §14)

Luật chọn y hệt nb12. Không arm nào bị giới hạn epoch (không có deferred), nên cả 5 arm
đều xét đủ 8 epoch.

In [ ]:
LOG = json.load(open(LOG_PATH))

def select_ckpt(arm):
    eps = {int(e): LOG[arm][e] for e in LOG[arm]}
    best = max(v['macro_map'] for v in eps.values())
    cand = [e for e, v in eps.items() if v['macro_map'] >= best - MACRO_TOL]
    key = lambda e: (eps[e]['tail_map'], eps[e]['worst3_fixed'], eps[e]['tail_recall_k'], eps[e]['macro_map'])
    pick = max(cand, key=key); pure = max(eps, key=lambda e: eps[e]['macro_map'])
    return dict(epoch=pick, metrics=eps[pick], snap=snap_path(arm, pick), n_cand=len(cand),
                best_macro=best, pure_epoch=pure, pure_metrics=eps[pure])

SEL = {t: select_ckpt(t) for t in ARMS}
print('CHON CHECKPOINT (§11): macro >= best-%.4f -> max tail mAP\n' % MACRO_TOL)
for t in ARMS:
    s = SEL[t]
    print(f"  {ARM_LABEL[t]:<22} ep{s['epoch']} (trong {s['n_cand']} ung vien, best macro "
          f"{s['best_macro']:.4f} @ep{s['pure_epoch']}) | tail {s['metrics']['tail_map']:.4f}")

COLS = [('macro_map','Macro mAP'), ('tail_map','Tail mAP'), ('worst3_fixed','Worst-3 fixed'),
        ('macro_auprc','Macro AUPRC'), ('macro_auroc','Macro AUROC'),
        ('tail_auroc','Tail AUROC'), ('tail_recall_k','Tail Recall@K')]
main = pd.DataFrame([{'Loss': ARM_LABEL[t], 'ep': SEL[t]['epoch'],
                      **{c[1]: SEL[t]['metrics'][c[0]] for c in COLS}} for t in ARMS])
base = SEL['A0']['metrics']
delta = main.copy()
for c in [c[1] for c in COLS]: delta[c] = main[c] - main[c].iloc[0]
print('\n=== BANG CHINH (§10) — VAL, study-level 7:3, seed 42 ===')
print(main.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print('\n=== DELTA vs A0 (ASL anchor cung notebook) ===')
print(delta.iloc[1:].to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
main.to_csv(f'{SC_DIR}/metrics/main_table.csv', index=False)
delta.to_csv(f'{SC_DIR}/metrics/delta_table.csv', index=False)

# ---------- A0 la regression test vs nb12 (§7) ----------
print('\n=== A0 regression check vs nb12 (phai gan trung) ===')
ref = NB12_REF['ASL']
for k, lab in [('macro_map','Macro mAP'), ('tail_map','Tail mAP')]:
    d = base[k] - ref[k]
    print(f'  {lab:<12} nb13 A0 {base[k]:.4f} (ep{SEL["A0"]["epoch"]}) vs nb12 ASL {ref[k]:.4f} '
          f'(ep{ref.get("epoch","?")}) | delta {d:+.4f}', '' if abs(d) < 0.002 else '  <-- LECH LON, kiem tra code-path')
_p12 = f'{NB12_DIR}/metrics/epoch_log.json'
if os.path.exists(_p12):
    L12 = json.load(open(_p12))
    g12 = lambda e: (L12['P'] if str(e) in L12.get('P', {}) else L12['L0'])[str(e)]['macro_map']
    print('  macro mAP theo epoch: nb13 A0 vs nb12 ASL')
    for e in range(1, EPOCHS+1):
        if str(e) in LOG['A0']:
            try: print(f'    ep{e}: {LOG["A0"][str(e)]["macro_map"]:.4f} vs {g12(e):.4f}'
                       f'  ({LOG["A0"][str(e)]["macro_map"]-g12(e):+.4f})')
            except KeyError: pass

### Quỹ đạo overfitting (§12) + so với WASL .50 (§14)

In [ ]:
traj = []
for t in ARMS:
    eps = sorted(int(e) for e in LOG[t])
    tail = {e: LOG[t][str(e)]['tail_map'] for e in eps}
    mac  = {e: LOG[t][str(e)]['macro_map'] for e in eps}
    bt, bm = max(tail, key=tail.get), max(mac, key=mac.get)
    traj.append(dict(arm=ARM_LABEL[t], best_tail=tail[bt], epoch_best_tail=bt, tail_ep8=tail[max(eps)],
                     tail_peak_drop=tail[bt]-tail[max(eps)], best_macro=mac[bm],
                     macro_ep8=mac[max(eps)], macro_peak_drop=mac[bm]-mac[max(eps)]))
traj = pd.DataFrame(traj)
print('=== ON DINH / OVERFIT (§12) ===')
print(traj.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print(f"\n  moc nb12 WASL .50: tail peak 0.1230 @ep2 -> ep8 0.1098 | drop 0.0132")
traj.to_csv(f'{SC_DIR}/metrics/stability.csv', index=False)

for metric, lab in [('tail_map','Tail mAP'), ('worst3_fixed','Worst-3 fixed'),
                    ('tail_recall_k','Tail Recall@K')]:
    tab = pd.DataFrame({ARM_LABEL[t]: {int(e): LOG[t][e][metric] for e in LOG[t]} for t in ARMS})
    print(f'\n--- {lab} theo epoch ---')
    print(tab.sort_index().to_string(float_format=lambda v: f'{v:.4f}'))
tl = pd.DataFrame({ARM_LABEL[t]: {int(e): LOG[t][e]['diag']['train_loss'] for e in LOG[t]} for t in ARMS})
print('\n--- train loss theo epoch (KHONG so sanh ngang giua loss khac ho) ---')
print(tl.sort_index().to_string(float_format=lambda v: f'{v:.5f}'))

print('\n--- grad norm trung binh (kiem tra scale optimization giua cac ho loss) ---')
gn = pd.DataFrame([dict(arm=ARM_LABEL[t], **{k: np.mean([LOG[t][e]['diag'][k] for e in LOG[t]
                        if LOG[t][e]['diag']['unfrozen']]) for k in
                        ('grad_decoder','grad_enc','grad_adapter')}) for t in ARMS])
print(gn.to_string(index=False, float_format=lambda v: f'{v:.5f}'))
_r = gn.set_index('arm').div(gn.set_index('arm').loc[ARM_LABEL['A0']])
print('  ty le vs ASL anchor:'); print(_r.to_string(float_format=lambda v: f'{v:.2f}'))
print('  LUU Y: neu ty le lech xa 1, mot phan chenh lech ket qua den tu SCALE gradient\n'
      '  (tuc LR hieu dung khac) chu khong chi tu thiet ke loss — §2 da canh bao.')

### Go / no-go (§13) + so sánh với WASL .50 (§14)

In [ ]:
print('=== GO / NO-GO vs ASL anchor (§13) ===\n')
verdict = {}
for t in ARMS[1:]:
    m = SEL[t]['metrics']
    d_tail = m['tail_map']-base['tail_map']; d_mac = m['macro_map']-base['macro_map']
    d_w3   = m['worst3_fixed']-base['worst3_fixed']; d_rk = m['tail_recall_k']-base['tail_recall_k']
    ok = (d_tail >= 0.002) and (d_mac >= -0.002) and (d_w3 > -0.002)
    verdict[t] = dict(d_tail_map=d_tail, d_macro_map=d_mac, d_worst3_fixed=d_w3,
                      d_tail_recall_k=d_rk, promising=bool(ok), prefer_recall=bool(d_rk >= 0))
    print(f'  {ARM_LABEL[t]:<22} dTail {d_tail:+.4f} | dMacro {d_mac:+.4f} | dW3 {d_w3:+.4f} | '
          f'dRecall@K {d_rk:+.4f}   {"PROMISING" if ok else "-"}')

W = NB12_REF['WASL .50']
print(f'\n=== vs WASL .50 cua nb12 (§14) — moc: Macro {W["macro_map"]:.4f} Tail {W["tail_map"]:.4f} '
      f'W3 {W["worst3_fixed"]:.4f} Rk {W["tail_recall_k"]:.4f} | tail collapse 0.0132 ===\n')
for t in ARMS[1:]:
    m = SEL[t]['metrics']; tr = traj[traj.arm == ARM_LABEL[t]].iloc[0]
    pA = (m['tail_map'] > W['tail_map'] + 0.002) and (m['macro_map'] >= base['macro_map'] - 0.002)
    pB = (abs(m['tail_map'] - W['tail_map']) <= 0.002 and
          m['worst3_fixed'] > W['worst3_fixed'] and m['tail_recall_k'] > W['tail_recall_k'] and
          tr.tail_peak_drop < 0.0132*0.6)
    tag = 'Pattern A (tail cao hon)' if pA else ('Pattern B (tail tuong duong, ben hon)' if pB else '-')
    print(f'  {ARM_LABEL[t]:<22} dTail_vs_WASL {m["tail_map"]-W["tail_map"]:+.4f} | '
          f'collapse {tr.tail_peak_drop:.4f} (WASL 0.0132) | {tag}')

any_go = any(v['promising'] for v in verdict.values())
print('\n' + ('KET LUAN: co arm qua cong -> can nhac Stage 1B (neg_scale in {2,3,5}) hoac Stage 2.'
      if any_go else
      'KET LUAN (§18 Case C): DB-Loss da duoc tune mapping van KHONG vuot ASL.\n'
      '  Ket luan manh hon truoc: khong phai do config cu te, ma la re-balancing theo phan phoi\n'
      '  khong giai quyet duoc nut that tail duoi kien truc RAD-DINO patch-token + EHR nay.\n'
      '  -> ha uu tien DBL.'))

json.dump(dict(selected={t: dict(epoch=SEL[t]['epoch'], snap=SEL[t]['snap'],
                                 **{k: SEL[t]['metrics'][k] for k in METRIC_KEYS},
                                 worst3_fixed=SEL[t]['metrics']['worst3_fixed']) for t in ARMS},
               worst3_fixed_labels=WORST3_FIXED, db_mu=DB_MU, verdict=verdict,
               any_promising=bool(any_go), stability=traj.to_dict('records'),
               nb12_ref=NB12_REF),
          open(f'{SC_DIR}/metrics/stage1a_result.json','w'), indent=1)
print('\nsaved -> metrics/stage1a_result.json, main_table.csv, delta_table.csv, stability.csv, '
      'db_weight_dist_*.csv, db_weight_pos_per_class.csv, db_config_audit.json')
print('\nSTOP (§19 buoc 10). Doc ket qua truoc; Stage 1B chi chay neu co arm qua cong.')

## C14 · Hybrid — `ASL + λ · tail-only DB-Loss`  (arm `H1`)

Ngoài prereg §7 của plan, chạy riêng, **không** thay đổi 5 arm đang chạy.

$$L \;=\; \underbrace{\text{mean}_{B\times C}\,\text{ASL}}_{\text{giữ nguyên toàn bộ 13 nhãn}}
\;+\;\lambda\cdot\underbrace{\text{mean}_{B\times|T|}\,\text{DB}}_{\text{chỉ cột tier tail}}$$

Ý tưởng: ep1 cho thấy DB-Loss toàn cục **phá** macro (−0.008…−0.015) nhưng lại **nâng
Tail Recall@K** (+0.005…+0.024). Nếu tín hiệu có ích của DB nằm ở phía tail thì áp DB **chỉ
lên cột tail** và cộng vào ASL sẽ giữ được macro của ASL mà vẫn lấy phần tail của DB.

Cột **không** thuộc tail có hệ số DB **đúng bằng 0**, nên gradient ở đó bằng ASL thuần
bit-for-bit — không phải xấp xỉ. Stage-0 assert điều này.

### Vì sao `μ = 0.70` chứ không phải 0.2

Tail ở cohort này chỉ có `Pleural_Other`. Với nhãn đó `r = freq_inv_{PO} / Σ_j y_j freq_inv_j`
dao động ~0.4–1.0 tuỳ số nhãn đồng xuất hiện. Với `β = 10`:

| μ | r=0.65 (đi kèm Fracture) | r=0.96 (đi kèm Pleural_Effusion) | spread |
|---:|---:|---:|---|
| 0.20 | 1.088 | 1.099 | ~0 → DB thoái hoá thành BCE-focal hằng số |
| **0.70** | **0.468** | **1.029** | rộng → cơ chế co-occurrence thật sự hoạt động |

Nói cách khác ở μ thấp thì "tail-only DBL" không còn là DBL nữa. μ=0.70 cũng là đỉnh
`tail/head` trong sweep, và ep1 cho thấy tail mAP tăng đơn điệu .10→.20→.30 rồi mới rơi ở .90.

### Chi phí

1 arm × 8 epoch. Backbone block 0–8 (`forward_shared`) chạy chung 1 lần/batch bất kể số arm,
nên 1 arm ≈ **35–45%** thời gian mỗi epoch của run 5-arm, không phải 20%.

`H1` so trực tiếp với `A0` của run chính: cùng seed, cùng `build_arm(42)`, cùng thứ tự batch
(`torch.manual_seed(SEED*7919+ep)`), cùng dropout mask (`sync_rng`) ⇒ **paired**, miễn là
`HY_BATCH` trùng `BATCH` của run chính.

In [ ]:
# ============ C14a · dinh nghia hybrid + Stage 0 (vai giay, khong train) ============
HY_LAMBDA = 0.25
HY_MU     = 0.70
HY_ARM    = 'H1'
HY_DIR    = os.path.join(DATA_DIR, 'hybrid_screen')
for _d in (HY_DIR, f'{HY_DIR}/checkpoints', f'{HY_DIR}/metrics', f'{HY_DIR}/snapshots'):
    os.makedirs(_d, exist_ok=True)
TAIL_J = np.where(TIER == 'tail')[0]

class HybridASLTailDB(nn.Module):
    """L = mean_{B,C} ASL + lam * mean_{B,|T|} DB(chi cot tail).

    Cai dat bang mot ma tran he so `mask` (=0 ngoai tail) sao cho el.mean() dung bang
    cong thuc tren -> giu duoc giao dien parts() nhu moi arm khac.
    """
    def __init__(s, lam=HY_LAMBDA, mu=HY_MU, tail_j=TAIL_J,
                 class_freq=None, train_num=None, **fixed):
        super().__init__()
        s.lam, s.mu, s.tail_j = lam, mu, np.asarray(tail_j)
        s.asl = ASLArm()
        s.db  = ResampleLoss(class_freq=class_freq, train_num=train_num, reduction='none',
                             loss_weight=fixed['loss_weight'], reweight_func=fixed['reweight_func'],
                             focal=fixed['focal'],
                             map_param=dict(alpha=fixed['map_alpha'], beta=fixed['map_beta'], gamma=mu),
                             logit_reg=dict(neg_scale=fixed['neg_scale'], init_bias=fixed['init_bias']))
        C = len(LABELS)
        m = torch.zeros(C)
        m[torch.as_tensor(s.tail_j, dtype=torch.long)] = lam * C / len(s.tail_j)
        s.register_buffer('mask', m)
    def _asl_el(s, x, y):
        a = s.asl; xp = torch.sigmoid(x); xn = 1 - xp
        if a.clip: xn = (xn + a.clip).clamp(max=1)
        lp = y*torch.log(xp.clamp(min=a.eps)); ln = (1-y)*torch.log(xn.clamp(min=a.eps))
        pt = xp*y + xn*(1-y); g = a.gp*y + a.gn*(1-y); mod = torch.pow(1-pt, g)
        return -lp*mod, -ln*mod
    def forward(s, x, y, parts=False):
        y = y.float()
        Lp, Ln = s._asl_el(x, y)
        add = s.mask[None, :] * s.db(x, y)
        loss = (Lp + Ln + add).mean()
        if parts:
            return loss, (Lp + add*y).mean(0).detach(), (Ln + add*(1-y)).mean(0).detach()
        return loss

_mk_hy = lambda lam, mu: HybridASLTailDB(lam, mu, TAIL_J, class_freq=CF, train_num=NTR_N,
                                         **DB_FIXED).to(dev)
print(f'tail cols: {[LABELS[j] for j in TAIL_J]} (idx {TAIL_J.tolist()}) | '
      f'lambda {HY_LAMBDA} | mu {HY_MU} | he so tren cot tail = lam*C/|T| = '
      f'{HY_LAMBDA*len(LABELS)/len(TAIL_J):.4f}')

# ---------------------------- STAGE 0 ----------------------------
torch.manual_seed(0)
_x = (torch.randn(128, len(LABELS), device=dev) * 3).requires_grad_(True)
_y = torch.zeros(128, len(LABELS), device=dev)
for _i in range(128):
    _y[_i, torch.randperm(len(LABELS), device=dev)[:int(torch.randint(1, 5, (1,)))]] = 1.
_y[:8, TAIL_J[0]] = 1.                                     # dam bao co du duong o cot tail

# H0: elementwise ASL trong hybrid == ASLArm (khong duoc lech dinh nghia)
_h = _mk_hy(0.0, HY_MU)
_a, _b = _h._asl_el(_x, _y)
print(f'\nH0  ASL elementwise: (Lp+Ln).mean() {(_a+_b).mean().item():.10f} vs ASLArm '
      f'{ASLArm().to(dev)(_x,_y).item():.10f}',
      'OK' if torch.equal((_a+_b).mean(), ASLArm().to(dev)(_x, _y)) else '!! LECH')
assert torch.equal((_a+_b).mean(), ASLArm().to(dev)(_x, _y))

# H1: lambda=0 -> bit-for-bit ASL thuan
print(f'H1  lambda=0 -> {_h(_x,_y).item():.10f} vs ASL {ASLArm().to(dev)(_x,_y).item():.10f}',
      'OK' if torch.equal(_h(_x, _y), ASLArm().to(dev)(_x, _y)) else '!! LECH')
assert torch.equal(_h(_x, _y), ASLArm().to(dev)(_x, _y))

# H2: loss == ASL + lam * mean(DB tren cot tail)
_h = _mk_hy(HY_LAMBDA, HY_MU)
_expect = ASLArm().to(dev)(_x, _y) + HY_LAMBDA * _h.db(_x, _y.float())[:, TAIL_J].mean()
print(f'H2  hybrid {_h(_x,_y).item():.10f} vs ASL + {HY_LAMBDA}*DB_tail {_expect.item():.10f} | '
      f'|diff| {abs(_h(_x,_y).item()-_expect.item()):.2e}')
assert torch.allclose(_h(_x, _y), _expect, atol=1e-7)

# H3: gradient o cot NGOAI tail phai bang ASL thuan CHINH XAC
_g_hy  = torch.autograd.grad(_h(_x, _y), _x, retain_graph=True)[0]
_g_asl = torch.autograd.grad(ASLArm().to(dev)(_x, _y), _x, retain_graph=True)[0]
_nt = np.setdiff1d(np.arange(len(LABELS)), TAIL_J)
_dn = (_g_hy[:, _nt] - _g_asl[:, _nt]).abs().max().item()
_dt = (_g_hy[:, TAIL_J] - _g_asl[:, TAIL_J]).abs().max().item()
print(f'H3  max|dgrad| cot NGOAI tail {_dn:.3e} (phai = 0) | cot TAIL {_dt:.3e} (phai > 0)',
      'OK' if _dn == 0.0 and _dt > 0 else '!! SAI')
assert _dn == 0.0 and _dt > 0

# H4: lambda mua duoc bao nhieu gradient tren cot tail?
_j = int(TAIL_J[0])
_g_db = (_g_hy[:, _j] - _g_asl[:, _j]).norm().item(); _g_a = _g_asl[:, _j].norm().item()
print(f'H4  tren cot "{LABELS[_j]}": ||grad tu phan DB|| / ||grad tu ASL|| = '
      f'{_g_db/_g_a:.3f}  (lam={HY_LAMBDA})')

# H5: DB weight tren cot tail co thuc su trai rong khong (co che co-occurrence con song)?
_Yt = torch.from_numpy(ehr.loc[(ehr.split=='train').to_numpy(), LABELS].to_numpy('float32')).to(dev)
_W  = torch.cat([_h.db.rebalance_weight(_Yt[i:i+20000]) for i in range(0, len(_Yt), 20000)])
_wt = _W[_Yt[:, _j] == 1, _j].cpu().numpy()
print(f'H5  w tren o DUONG cua "{LABELS[_j]}" (n={_wt.size:,}): min {_wt.min():.4f} '
      f'p05 {np.percentile(_wt,5):.4f} median {np.median(_wt):.4f} mean {_wt.mean():.4f} '
      f'p95 {np.percentile(_wt,95):.4f} max {_wt.max():.4f}')
print(f'    spread p95-p05 = {np.percentile(_wt,95)-np.percentile(_wt,5):.4f} '
      f'(gan 0 => mu qua thap, DB thoai hoa thanh hang so -> chinh HY_MU)')
_nco = int((_Yt[_Yt[:, _j] == 1].sum(1) == 1).sum())
print(f'    {_nco:,}/{_wt.size:,} study co "{LABELS[_j]}" la nhan duong DUY NHAT (r=1 -> w cao nhat)')
del _Yt, _W; torch.cuda.empty_cache()
print('\nSTAGE 0 HYBRID PASS')

### C14b · Train `H1` (1 arm × 8 epoch)

`HY_BATCH` **phải** bằng `BATCH` của run chính, nếu không thứ tự batch khác đi và mất tính
paired với `A0`. Cell tự lấy biến `BATCH` trong session và in ra để đối chiếu.

Resume độc lập (`hybrid_screen/checkpoints/_resume.pt`), không đụng `dbl_screen/`.

In [ ]:
# ============ C14b · train arm H1 (chay sau khi run chinh xong, hoac song song neu du VRAM) ============
HY_BATCH = BATCH                       # PHAI trung run chinh -> giu tinh paired voi A0
HY_LOG_PATH = f'{HY_DIR}/metrics/epoch_log.json'
hy_snap   = lambda ep: f'{HY_DIR}/snapshots/{HY_ARM}_ep{ep}.pt'
hy_resume = lambda: f'{HY_DIR}/checkpoints/_resume.pt'
print(f'HY_BATCH = {HY_BATCH} (BATCH run chinh) | accum {max(1, EFF//HY_BATCH)} -> effective {EFF}')

def train_hybrid(batch=HY_BATCH, epochs=EPOCHS):
    dl    = loader('train', train_tf, True, batch)
    accum = max(1, EFF//batch)
    TOTAL = epochs*math.ceil(len(dl)/accum); WARM = int(PREREG['warmup_frac']*TOTAL)
    crit  = _mk_hy(HY_LAMBDA, HY_MU)
    LOG = json.load(open(HY_LOG_PATH)) if os.path.exists(HY_LOG_PATH) else {}
    if os.path.exists(hy_resume()):
        c = torch.load(hy_resume(), map_location=dev, weights_only=False)
        if c['batch'] != batch: raise RuntimeError(f"resume batch {c['batch']} != {batch}")
        if (c.get('lam'), c.get('mu')) != (HY_LAMBDA, HY_MU):
            raise RuntimeError(f"resume la (lam,mu)={c.get('lam'),c.get('mu')} != {(HY_LAMBDA,HY_MU)}; "
                               f"doi HY_DIR hoac xoa {hy_resume()}")
        m = ArmModel().to(dev); m.load_state_dict(c['model'])
        opt = make_opt(m); opt.load_state_dict(c['opt'])
        sch = make_sched(opt, TOTAL, WARM); sch.load_state_dict(c['sch'])
        start = c['epoch']+1; print(f'RESUMED H1 @ep{start+1}')
        if start >= epochs: print('H1 da chay xong'); return LOG
    else:
        m = build_arm(SEED); opt = make_opt(m); sch = make_sched(opt, TOTAL, WARM); start = 0
        print(f'init H1 = build_arm({SEED}) (trung init cua A0) | steps/epoch '
              f'{math.ceil(len(dl)/accum)} | total {TOTAL} | warmup {WARM}')

    A0 = json.load(open(f'{SC_DIR}/metrics/epoch_log.json')).get('A0', {}) \
         if os.path.exists(f'{SC_DIR}/metrics/epoch_log.json') else {}
    for ep in range(start, epochs):
        unfrozen = ep >= DECODER_WARMUP_EPOCHS; tag = '[dec]' if unfrozen else '[warm]'
        set_decoder_trainable(m, unfrozen); m.train()
        z0 = lambda: torch.zeros(len(LABELS), device=dev)
        d = dict(loss=torch.zeros((), device=dev), n=0, pos=z0(), neg=z0(),
                 gd=torch.zeros((), device=dev), ge=torch.zeros((), device=dev),
                 ga=torch.zeros((), device=dev), nopt=0)
        torch.manual_seed(SEED*7919 + ep)                    # thu tu batch: trung run chinh
        for i, b in enumerate(tqdm(dl, desc=f'H1 ep{ep+1}/{epochs}{tag}', leave=False)):
            b = to_dev(b); z = forward_shared(b['image'])
            sync_rng(ep, i)                                  # dropout mask: trung run chinh
            loss, lp, ln = crit(m(z, b), b['y'], parts=True)
            full = loss + L2SP_DECODER*l2sp_term(m) if unfrozen else loss
            (full/accum).backward()
            d['loss'] += loss.detach(); d['pos'] += lp; d['neg'] += ln; d['n'] += 1
            if ((i+1) % accum == 0) or (i == len(dl)-1):
                gd, ge, ga = grad_norms(m)
                d['gd'] += gd; d['ge'] += ge; d['ga'] += ga; d['nopt'] += 1
                clip_arm(m); opt.step(); sch.step(); opt.zero_grad(set_to_none=True)

        r = eval_all(predict({HY_ARM: m}, 'val', batch*2)[HY_ARM], 'val')
        r['worst3_fixed'] = w3_fixed(r)
        n, no = max(d['n'],1), max(d['nopt'],1)
        pos = (d['pos']/n).cpu().numpy(); neg = (d['neg']/n).cpu().numpy()
        tm = lambda v: {k: float(v[TIER==k].mean()) for k in ('head','medium','tail')}
        r['diag'] = dict(train_loss=float(d['loss'].item()/n), pos=tm(pos), neg=tm(neg),
                         grad_decoder=float(d['gd'].item()/no), grad_enc=float(d['ge'].item()/no),
                         grad_adapter=float(d['ga'].item()/no),
                         lr_adapter=opt.param_groups[0]['lr'], unfrozen=bool(unfrozen))
        r['hybrid'] = dict(lam=HY_LAMBDA, mu=HY_MU, tail_cols=[LABELS[j] for j in TAIL_J])
        LOG.setdefault(HY_ARM, {})[str(ep+1)] = r
        torch.save({k: v.detach().cpu() for k, v in m.state_dict().items()}, hy_snap(ep+1))
        json.dump(LOG, open(HY_LOG_PATH,'w'), indent=1)
        torch.save(dict(model=m.state_dict(), opt=opt.state_dict(), sch=sch.state_dict(),
                        epoch=ep, batch=batch, lam=HY_LAMBDA, mu=HY_MU), hy_resume()+'.tmp')
        os.replace(hy_resume()+'.tmp', hy_resume())
        line = (f"  ep{ep+1}{tag} H1 mAP {r['macro_map']:.4f} tail {r['tail_map']:.4f} "
                f"w3f {r['worst3_fixed']:.4f} rk {r['tail_recall_k']:.3f} L {r['diag']['train_loss']:.4f}")
        if str(ep+1) in A0:
            a = A0[str(ep+1)]
            line += (f"  | vs A0: dmAP {r['macro_map']-a['macro_map']:+.4f} "
                     f"dtail {r['tail_map']-a['tail_map']:+.4f} "
                     f"dw3f {r['worst3_fixed']-a['worst3_fixed']:+.4f} "
                     f"drk {r['tail_recall_k']-a['tail_recall_k']:+.3f}")
        print(line + '  [saved]')
    return LOG

HY_LOG = train_hybrid()

# ---------------------------- ket qua: H1 vs A0 (paired) ----------------------------
_a0p = f'{SC_DIR}/metrics/epoch_log.json'
if os.path.exists(_a0p) and 'A0' in json.load(open(_a0p)):
    A0 = json.load(open(_a0p))['A0']
    eps = sorted(set(int(e) for e in HY_LOG[HY_ARM]) & set(int(e) for e in A0))
    cmp = pd.DataFrame([dict(ep=e,
            A0_macro=A0[str(e)]['macro_map'], H1_macro=HY_LOG[HY_ARM][str(e)]['macro_map'],
            d_macro=HY_LOG[HY_ARM][str(e)]['macro_map']-A0[str(e)]['macro_map'],
            A0_tail=A0[str(e)]['tail_map'],   H1_tail=HY_LOG[HY_ARM][str(e)]['tail_map'],
            d_tail=HY_LOG[HY_ARM][str(e)]['tail_map']-A0[str(e)]['tail_map'],
            A0_w3f=A0[str(e)]['worst3_fixed'], H1_w3f=HY_LOG[HY_ARM][str(e)]['worst3_fixed'],
            d_w3f=HY_LOG[HY_ARM][str(e)]['worst3_fixed']-A0[str(e)]['worst3_fixed'],
            A0_rk=A0[str(e)]['tail_recall_k'], H1_rk=HY_LOG[HY_ARM][str(e)]['tail_recall_k'],
            d_rk=HY_LOG[HY_ARM][str(e)]['tail_recall_k']-A0[str(e)]['tail_recall_k'])
            for e in eps])
    print(f'\n=== H1 (ASL + {HY_LAMBDA}*tail-DBL mu={HY_MU}) vs A0 (ASL) — paired, tung epoch ===')
    print(cmp.to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
    cmp.to_csv(f'{HY_DIR}/metrics/h1_vs_a0.csv', index=False)

    def _sel(log):
        e = {int(k): v for k, v in log.items()}
        best = max(v['macro_map'] for v in e.values())
        cand = [k for k, v in e.items() if v['macro_map'] >= best - MACRO_TOL]
        return max(cand, key=lambda k: (e[k]['tail_map'], e[k]['worst3_fixed'],
                                        e[k]['tail_recall_k'], e[k]['macro_map'])), e
    eh, EH = _sel(HY_LOG[HY_ARM]); ea, EA = _sel(A0)
    print(f'\n=== checkpoint chon theo §11 ===')
    for nm, ep_, E in (('A0 ASL', ea, EA), (f'H1 hybrid', eh, EH)):
        m_ = E[ep_]
        print(f'  {nm:<12} ep{ep_} | macro {m_["macro_map"]:.4f} tail {m_["tail_map"]:.4f} '
              f'w3f {m_["worst3_fixed"]:.4f} rk {m_["tail_recall_k"]:.4f}')
    dt = EH[eh]['tail_map']-EA[ea]['tail_map']; dm = EH[eh]['macro_map']-EA[ea]['macro_map']
    dw = EH[eh]['worst3_fixed']-EA[ea]['worst3_fixed']
    print(f'\n  dTail {dt:+.4f} | dMacro {dm:+.4f} | dW3f {dw:+.4f}  -> '
          f'{"PROMISING (cong §13)" if (dt>=0.002 and dm>=-0.002 and dw>-0.002) else "khong qua cong §13"}')
    W = NB12_REF['WASL .50']
    print(f'  vs WASL .50 (nb12): dTail {EH[eh]["tail_map"]-W["tail_map"]:+.4f} | '
          f'dMacro {EH[eh]["macro_map"]-W["macro_map"]:+.4f}')
    print('\n  LUU Y: H1 nam NGOAI prereg §7 (exploratory). Ket qua duong tinh can replicate\n'
          '  o seed khac truoc khi coi la ket luan.')
else:
    print('\nchua co A0 trong dbl_screen/metrics/epoch_log.json -> chay run chinh truoc de co moc paired')